In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [7]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root='./data',train=True,download=True,transform=transform)
testset = CIFAR10(root='./data',train=False,download=True,transform=transform)

100%|████████████████████████████████████████| 170M/170M [00:24<00:00, 6.95MB/s]


In [8]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(testset,batch_size = 64)

In [21]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) ,

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening
        x = self.fc_layers(x)

        return x

In [22]:
model = CNN()

In [23]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [25]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=0.9356468720814152
epoch=2/10 & loss=0.7495584470384261
epoch=3/10 & loss=0.625324187566862
epoch=4/10 & loss=0.5184090219800125
epoch=5/10 & loss=0.423844149522007
epoch=6/10 & loss=0.33815966072061177
epoch=7/10 & loss=0.2670275778950328
epoch=8/10 & loss=0.20525973646537118
epoch=9/10 & loss=0.16532004860413196
epoch=10/10 & loss=0.13215912668072544


In [ ]:
# Evaluate our CNN

correct_label = 0
total_label = 0

model.eval()

with torch.no_grad():
    for images,labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs,1)

        correct_labels +=(predicted == labels).sum().item()
        total_labels+=labels.size(0)


print(f'accuracy={correct_labels/total_labels*100}')